In [1]:
import os
import sys
base_directory = os.path.dirname(os.path.abspath(""))
sys.path.append(base_directory)
from src.build_database_table import extract_with_agent

# import importlib
# import src.build_database_table as bdt
# importlib.reload(bdt)
# from src.build_database_table import extract_with_agent

gpu_id = 0
vllm_port = 8001
model = "Qwen/Qwen2.5-7B-Instruct"

2026-03-24 15:02:40,349 INFO | Using Entrez API key from environment variable.


# Setting up vLLM

In [3]:
# check if vllm running in port {vllm_port}, if not, start it
import socket
def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

if not is_port_in_use(vllm_port):
    !conda run -n vllm CUDA_VISIBLE_DEVICES={gpu_id} python -m vllm.entrypoints.openai.api_server --model {model} --port {vllm_port} --enforce-eager --enable-auto-tool-choice --tool-call-parser hermes > vllm.log 2>&1 &

# Test run

In [4]:
title = "Merlin: a computed tomography vision–language foundation model and dataset"
abstract = "The large volume of abdominal computed tomography (CT) scans1,2 coupled with the shortage of radiologists3,4,5,6 have intensified the need for automated medical image analysis tools. Previous state-of-the-art approaches for automated analysis leverage vision–language models (VLMs) that jointly model images and radiology reports7,8,9,10,11,12. However, current medical VLMs are generally limited to 2D images and short reports. Here to overcome these shortcomings for abdominal CT interpretation, we introduce Merlin, a 3D VLM that learns from volumetric CT scans, electronic health record data and radiology reports. This approach is enabled by a multistage pretraining framework that does not require additional manual annotations. We trained Merlin using a high-quality clinical dataset of paired CT scans (>6 million images from 15,331 CT scans), diagnosis codes (>1.8 million codes) and radiology reports (>6 million tokens). We comprehensively evaluated Merlin on 6 task types and 752 individual tasks that covered diagnostic, prognostic and quality-related tasks. The non-adapted (off-the-shelf) tasks included zero-shot classification of findings (30 findings), phenotype classification (692 phenotypes) and zero-shot cross-modal retrieval (image-to-findings and image-to-impression). The model-adapted tasks included 5-year chronic disease prediction (6 diseases), radiology report generation and 3D semantic segmentation (20 organs). We validated Merlin at scale, with internal testing on 5,137 CT scans and external testing on 44,098 CT scans from 3 independent sites and 2 public datasets. The results demonstrated high generalization across institutions and anatomies. Merlin outperformed 2D VLMs, CT foundation models and off-the-shelf radiology models. We also computed scaling laws and conducted ablation studies to identify optimal training strategies. We release our trained models, code and dataset for 25,494 pairs of abdominal CT scans and radiology reports. Our results demonstrate how Merlin may assist in the interpretation of abdominal CT scans and mitigate the burden on radiologists while simultaneously adding value for future biomarker discovery and disease risk stratification."
link = "https://doi.org/10.1038/s41586-026-10181-8"

dataset = await extract_with_agent(title, abstract, publication_metadata={"link": link})
print(dataset)

name='Merlin' num_images=25494 num_patients=15331 modalities=[<Modality.CT: 'CT'>] body_regions=[<BodyRegion.ABDOMEN: 'abdomen'>] additional_data=[<AdditionalData.REPORTS: 'reports'>] paper_title='Merlin: a computed tomography vision–language foundation model and dataset' paper_link='https://doi.org/10.1038/s41586-026-10181-8' paper_year=None paper_authors=None paper_journal=None pmid=None paper_citation_count=None


# Run the script

In [ ]:
!python3 {base_directory}/src/build_database_table.py

72.85s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Traceback (most recent call last):
  File "/home/jrich/Desktop/radiology_dataset_db/src/build_database_table.py", line 22, in <module>
    Entrez.email = getenv("ENTREZ_EMAIL")
  File "/home/jrich/Desktop/radiology_dataset_db/src/build_database_table.py", line 19, in getenv
    raise ValueError(f"Environment variable {key} is not set.")
ValueError: Environment variable ENTREZ_EMAIL is not set.


# End the vLLM server

In [ ]:
# check if it's running
if is_port_in_use(vllm_port):
    print(f"vLLM is running on port {vllm_port}")
    !pkill -f vllm

In [ ]:
!pip list